# Exploratory Data Analysis - IPL Auction Price Project

Exploratory Data Analysis - IPL auction price project
Run this file once. It reads the merged dataset and produces, in an output
folder:
    - summary statistic tables saved as .csv (paste into the thesis)
    - figures saved as .png (attach in the thesis)

The code is grouped into six sections, in the order you would present them:
    1. Price distribution        -> justifies the log transform
    2. Season dimension          -> price level by auction year
    3. Sold vs unsold            -> classifier target + selection-bias story
    4. Player characteristics    -> role-conditional summaries
    5. Price vs features          -> sets up the hedonic regression
    6. Missingness / data quality -> the "data limitations" section

Every plot follows the same simple pattern: make a figure, draw on it,
label it, save it. No shortcuts, so each block is easy to explain.

## Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
from scipy import stats

## CONFIG  ->  change these two paths to match your machine

In [ ]:
# Absolute paths in your usual style. Edit the two lines to match your machine.
# (A relative path like r"..\processed_data\..." also works if you run this
#  file from your pythoncodes folder with processed_data as a sibling.)
DATA_PATH = r"D:\DataEngineering\Final Year Project\processed_data\final_modelling_ready_dataset.csv"
OUTPUT_DIR = r"D:\DataEngineering\Final Year Project\processed_data\eda_outputs"

# Create the output folder if it does not exist yet
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Make the figures look clean (falls back to default if the style is missing)
try:
    plt.style.use("seaborn-v0_8-whitegrid")
except Exception:
    pass
plt.rcParams["savefig.dpi"] = 200      # high resolution for the thesis
plt.rcParams["font.size"] = 11

## Small helper: save a figure and close it (keeps the code below tidy)

In [ ]:
def save(fig, filename):
    path = os.path.join(OUTPUT_DIR, filename)
    fig.savefig(path, bbox_inches="tight")
    plt.show()
    print("saved figure:", filename)

## STEP 0 - Load the data and define the analysis populations

In [ ]:
df = pd.read_csv(DATA_PATH)
print("Full dataset shape:", df.shape)

# Price in crore (1 crore = 10,000,000 rupees), computed transparently
df["price_cr"] = df["sold_price"] / 1e7

# Log of the raw price, defined only where a positive price exists
df["log_price"] = np.where(df["sold_price"] > 0, np.log(df["sold_price"]), np.nan)

# The 'age' column is text like '43y 78d' and is the player's CURRENT age,
# not their age at the auction. For a price model we want age AT the auction,
# so we compute it from the birthdate and the auction year instead.
birth = pd.to_datetime(df["birthdate"], errors="coerce")
df["age_at_auction"] = df["year"] - birth.dt.year

# Population A: rows that have a real, positive price (sold + retained)
priced = df[df["sold_price"].notna() & (df["sold_price"] > 0)].copy()

# Population B: players who went through the open auction (sold or unsold)
auctioned = df[df["auction_result"].isin(["sold", "unsold"])].copy()

print("Priced rows   :", len(priced))
print("Auctioned rows:", len(auctioned))

## SECTION 1 - PRICE DISTRIBUTION  (justifies the log transform)

In [ ]:
# 1a. Summary statistics of the price, in crore
price_summary = priced["price_cr"].describe()
price_summary["median"] = priced["price_cr"].median()
price_summary["skew_raw"] = priced["sold_price"].skew()
price_summary["skew_log"] = priced["log_price"].skew()
price_summary.to_csv(os.path.join(OUTPUT_DIR, "01_price_summary.csv"))
print("\n--- Section 1: price summary (crore) ---")
print(price_summary.round(3))

In [ ]:
# 1b. Two histograms side by side: raw price vs log price
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].hist(priced["price_cr"].dropna(), bins=40, color="#4C72B0", edgecolor="white")
axes[0].set_title("Raw sold price (right-skewed)")
axes[0].set_xlabel("Sold price (crore)")
axes[0].set_ylabel("Number of players")

axes[1].hist(priced["log_price"].dropna(), bins=40, color="#55A868", edgecolor="white")
axes[1].set_title("Log of sold price (symmetric)")
axes[1].set_xlabel("log(sold price)")
axes[1].set_ylabel("Number of players")

fig.suptitle("Distribution of IPL auction prices", fontsize=13)
save(fig, "01_price_distribution.png")

In [ ]:
# 1c. Q-Q plot of log price against a normal distribution
fig, ax = plt.subplots(figsize=(5.5, 5.5))
stats.probplot(priced["log_price"].dropna(), dist="norm", plot=ax)
ax.set_title("Q-Q plot: log price vs normal")
save(fig, "01_qqplot_log_price.png")

In [ ]:
# 1d. Ten most expensive buys (gives the reader a feel for the tail)
top10 = priced.sort_values("price_cr", ascending=False).head(10)
top10_table = top10[["year", "player_name", "team", "price_cr", "player_role"]]
top10_table.to_csv(os.path.join(OUTPUT_DIR, "01_top10_expensive.csv"), index=False)
print("\nTop 10 most expensive buys:")
print(top10_table.to_string(index=False))

## SECTION 2 - SEASON DIMENSION  (price level by auction year)

In [ ]:
# 2a. Number of priced players per auction year
counts_per_year = priced.groupby("year").size()

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.bar(counts_per_year.index, counts_per_year.values, color="#4C72B0", edgecolor="white")
ax.set_title("Number of priced players per auction year")
ax.set_xlabel("Auction year")
ax.set_ylabel("Number of players")
ax.set_xticks(counts_per_year.index)
ax.tick_params(axis="x", rotation=45)
save(fig, "02_players_per_year.png")

In [ ]:
# 2b. Median and mean price per year (the headline: prices drift upward)
median_by_year = priced.groupby("year")["price_cr"].median()
mean_by_year = priced.groupby("year")["price_cr"].mean()

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(median_by_year.index, median_by_year.values, marker="o", label="Median")
ax.plot(mean_by_year.index, mean_by_year.values, marker="s", label="Mean")
ax.set_title("Sold price by auction year")
ax.set_xlabel("Auction year")
ax.set_ylabel("Sold price (crore)")
ax.set_xticks(median_by_year.index)
ax.tick_params(axis="x", rotation=45)
ax.legend()
save(fig, "02_price_by_year.png")

In [ ]:
# 2c. Box plot of LOG price per year (shows the level shifting up AND the
#     large spread within each single auction)
years_sorted = sorted(priced["year"].unique())
data_per_year = []
for y in years_sorted:
    values = priced.loc[priced["year"] == y, "log_price"].dropna().values
    data_per_year.append(values)

fig, ax = plt.subplots(figsize=(12, 5))
ax.boxplot(data_per_year, showfliers=True)
ax.set_xticks(range(1, len(years_sorted) + 1))
ax.set_xticklabels(years_sorted, rotation=45)
ax.set_title("Distribution of log price within each auction year")
ax.set_xlabel("Auction year")
ax.set_ylabel("log(sold price)")
save(fig, "02_logprice_boxplot_by_year.png")

## SECTION 3 - SOLD VS UNSOLD  (classifier target + selection-bias story)

In [ ]:
# 3a. Sold vs unsold counts per year (stacked bars)
result_by_year = auctioned.groupby(["year", "auction_result"]).size().unstack(fill_value=0)
for col in ["sold", "unsold"]:
    if col not in result_by_year.columns:
        result_by_year[col] = 0

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.bar(result_by_year.index, result_by_year["sold"], label="Sold", color="#55A868")
ax.bar(result_by_year.index, result_by_year["unsold"],
       bottom=result_by_year["sold"], label="Unsold", color="#C44E52")
ax.set_title("Sold vs unsold players per auction year")
ax.set_xlabel("Auction year")
ax.set_ylabel("Number of players")
ax.set_xticks(result_by_year.index)
ax.tick_params(axis="x", rotation=45)
ax.legend()
save(fig, "03_sold_unsold_counts.png")

In [ ]:
# 3b. Proportion of players sold per year
sold_rate = result_by_year["sold"] / (result_by_year["sold"] + result_by_year["unsold"])

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(sold_rate.index, sold_rate.values, marker="o", color="#4C72B0")
ax.set_ylim(0, 1)
ax.set_title("Proportion of players sold per auction year")
ax.set_xlabel("Auction year")
ax.set_ylabel("Sold rate")
ax.set_xticks(sold_rate.index)
ax.tick_params(axis="x", rotation=45)
save(fig, "03_sold_rate.png")

In [ ]:
# 3c. Compare sold vs unsold players on their characteristics.
#     If the two groups differ, being sold is NOT random -> selection bias.
compare_cols = ["age_at_auction", "base_price", "matches", "runs", "strikerate", "wickets"]
group_means = auctioned.groupby("auction_result")[compare_cols].mean()
group_means.to_csv(os.path.join(OUTPUT_DIR, "03_sold_vs_unsold_means.csv"))
print("\n--- Section 3: mean characteristics, sold vs unsold ---")
print(group_means.round(2))

# Share capped and share overseas among sold vs unsold
share_table = auctioned.groupby("auction_result").agg(
    share_capped=("is_capped", "mean"),
    share_overseas=("overseas", "mean"),
    n=("is_capped", "size"),
)
share_table.to_csv(os.path.join(OUTPUT_DIR, "03_sold_vs_unsold_shares.csv"))
print("\nShare capped / overseas, sold vs unsold:")
print(share_table.round(3))

## SECTION 4 - PLAYER CHARACTERISTICS  (role-conditional)

In [ ]:
# 4a. Distributions of the key numeric features (one grid of histograms).
#     The title of each shows n, because these columns are often missing.
feature_cols = ["age_at_auction", "matches", "runs", "strikerate", "wickets", "economy"]

fig, axes = plt.subplots(2, 3, figsize=(13, 7))
axes = axes.flatten()
for i in range(len(feature_cols)):
    col = feature_cols[i]
    values = priced[col].dropna()
    axes[i].hist(values, bins=30, color="#4C72B0", edgecolor="white")
    axes[i].set_title(col + "  (n=" + str(len(values)) + ")")
fig.suptitle("Distributions of key player features (priced players)", fontsize=13)
save(fig, "04_feature_distributions.png")

In [ ]:
# 4b. Feature averages BY ROLE. A single overall mean mixes batsmen and
#     bowlers, so we split by role to make the numbers meaningful.
role_stats = priced.groupby("player_role")[feature_cols].mean()
role_stats["n_players"] = priced.groupby("player_role").size()
role_stats.to_csv(os.path.join(OUTPUT_DIR, "04_feature_means_by_role.csv"))
print("\n--- Section 4: feature means by role ---")
print(role_stats.round(2))

In [ ]:
# 4c. Counts of the categorical variables
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

role_counts = priced["player_role"].value_counts()
axes[0, 0].bar(role_counts.index, role_counts.values, color="#4C72B0")
axes[0, 0].set_title("Player role")
axes[0, 0].tick_params(axis="x", rotation=40)

capped_counts = priced["is_capped"].value_counts()
axes[0, 1].bar(capped_counts.index.astype(str), capped_counts.values, color="#55A868")
axes[0, 1].set_title("Capped (international) status")

status_counts = df["status"].value_counts()
axes[1, 0].bar(status_counts.index.astype(str), status_counts.values, color="#C44E52")
axes[1, 0].set_title("Status (New / Retained / RTM)")

result_counts = auctioned["auction_result"].value_counts()
axes[1, 1].bar(result_counts.index.astype(str), result_counts.values, color="#8172B3")
axes[1, 1].set_title("Auction result")

fig.suptitle("Counts of categorical variables", fontsize=13)
# labels can overlap, so give the bottom row a little room
for ax in axes.flatten():
    ax.tick_params(axis="x", labelsize=9)
save(fig, "04_categorical_counts.png")

## SECTION 5 - PRICE VS FEATURES  (sets up the hedonic regression)

In [ ]:
# 5a. Scatter of log price against each key feature
scatter_cols = ["age_at_auction", "matches", "runs", "strikerate", "wickets", "base_price"]

fig, axes = plt.subplots(2, 3, figsize=(13, 7))
axes = axes.flatten()
for i in range(len(scatter_cols)):
    col = scatter_cols[i]
    sub = priced[[col, "log_price"]].dropna()
    axes[i].scatter(sub[col], sub["log_price"], s=10, alpha=0.4, color="#4C72B0")
    axes[i].set_xlabel(col)
    axes[i].set_ylabel("log(price)")
fig.suptitle("Log price vs player features (priced players)", fontsize=13)
save(fig, "05_price_vs_features.png")

In [ ]:
# 5b. Log price by role, by capped status, and by overseas status
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# by role
roles = list(priced["player_role"].dropna().unique())
role_data = []
for r in roles:
    role_data.append(priced.loc[priced["player_role"] == r, "log_price"].dropna().values)
axes[0].boxplot(role_data)
axes[0].set_xticks(range(1, len(roles) + 1))
axes[0].set_xticklabels(roles, rotation=40, ha="right")
axes[0].set_title("Log price by role")
axes[0].set_ylabel("log(price)")

# by capped status
capped_data = []
capped_labels = ["Uncapped", "Capped"]
for v in [False, True]:
    capped_data.append(priced.loc[priced["is_capped"] == v, "log_price"].dropna().values)
axes[1].boxplot(capped_data)
axes[1].set_xticks([1, 2])
axes[1].set_xticklabels(capped_labels)
axes[1].set_title("Log price by capped status")

# by overseas status (1.0 = overseas, 0.0 = domestic)
overseas_data = []
overseas_labels = ["Domestic", "Overseas"]
for v in [0.0, 1.0]:
    overseas_data.append(priced.loc[priced["overseas"] == v, "log_price"].dropna().values)
axes[2].boxplot(overseas_data)
axes[2].set_xticks([1, 2])
axes[2].set_xticklabels(overseas_labels)
axes[2].set_title("Log price by overseas status")

save(fig, "05_price_by_category.png")

In [ ]:
# 5c. Correlation heatmap of the numeric features together with log price.
#     Useful for spotting which features track price and which move together
#     (multicollinearity, e.g. matches / innings / runs).
corr_cols = ["log_price", "base_price", "age_at_auction", "matches", "innings",
             "runs", "strikerate", "batting_average", "wickets", "economy"]
corr = priced[corr_cols].corr()

fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(corr.values, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr_cols)))
ax.set_yticks(range(len(corr_cols)))
ax.set_xticklabels(corr_cols, rotation=45, ha="right")
ax.set_yticklabels(corr_cols)
# write each correlation value inside its cell
for i in range(len(corr_cols)):
    for j in range(len(corr_cols)):
        ax.text(j, i, round(corr.values[i, j], 2),
                ha="center", va="center", color="black", fontsize=8)
fig.colorbar(im, ax=ax, shrink=0.8, label="Correlation")
ax.set_title("Correlation between numeric features")
save(fig, "05_correlation_heatmap.png")

## SECTION 6 - MISSINGNESS AND DATA QUALITY  (the "limitations" section)

In [ ]:
# 6a. Percent missing per column, across the whole dataset
missing_pct = df.isna().mean() * 100
missing_pct = missing_pct.sort_values(ascending=False)
missing_pct.to_csv(os.path.join(OUTPUT_DIR, "06_missingness_percent.csv"))
print("\n--- Section 6: top 15 columns by percent missing ---")
print(missing_pct.head(15).round(1))

top_missing = missing_pct.head(20)
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top_missing.index[::-1], top_missing.values[::-1], color="#C44E52")
ax.set_title("Percent missing (top 20 columns)")
ax.set_xlabel("Percent missing")
save(fig, "06_missingness.png")

In [ ]:
# 6b. Structural vs data-quality missingness: a lot of "missing" stats are
#     simply players with no prior season, which is missing BY DESIGN.
stat_cols = ["runs", "wickets", "matches", "strikerate"]
rows = []
for flag, group in df.groupby("has_prior_season"):
    pct = group[stat_cols].isna().mean() * 100
    pct.name = "has_prior_season=" + str(flag)
    rows.append(pct)
missing_by_prior = pd.DataFrame(rows)
missing_by_prior.to_csv(os.path.join(OUTPUT_DIR, "06_missing_by_prior_season.csv"))
print("\nPercent missing in stat columns, split by has_prior_season:")
print(missing_by_prior.round(1))

In [ ]:
# 6c. Known integrity issues, as counts you can quote in the thesis.
#     NOTE: a player_id repeating across years is EXPECTED (same player,
#     many auctions), so we do NOT count that. We count real problems:
n_missing_id = df["player_id"].isna().sum()

# one name mapped to more than one id (e.g. a trailing-digit typo)
id_per_name = df.groupby("player_name")["player_id"].nunique()
n_name_multi_id = (id_per_name > 1).sum()

# one id mapped to more than one name (an upper bound: partly caused by
# encoding-damaged name strings for the SAME player, so inspect before quoting)
name_per_id = df.dropna(subset=["player_id"]).groupby("player_id")["player_name"].nunique()
n_id_multi_name = (name_per_id > 1).sum()

issues = pd.DataFrame({
    "issue": [
        "rows with missing player_id",
        "names mapped to >1 different id",
        "ids mapped to >1 different name (upper bound; check encoding)",
    ],
    "count": [n_missing_id, n_name_multi_id, n_id_multi_name],
})
issues.to_csv(os.path.join(OUTPUT_DIR, "06_integrity_issues.csv"), index=False)
print("\nKnown integrity issues:")
print(issues.to_string(index=False))

print("\nDONE. All tables and figures are in:", OUTPUT_DIR)